# Casing & Preprocessing OOD Experiment

**DS-UA 301** | Wendy Liu, Wency Li, Yujia Guo

Three experiments to attribute the OOD failure (sentence-cased human reviews classified as AI at score ≈ 100):
- **A** — does removing casing from AI reviews flip predictions?
- **B** — does sentence-casing human reviews flip predictions?
- **C** — stepwise preprocessing attribution.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

notebook_dir = Path.cwd()
repo_root = None
for candidate in [notebook_dir, *notebook_dir.parents]:
    if (candidate / "src" / "final_detector.py").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Could not locate src/final_detector.py")

sys.path.insert(0, str(repo_root))

from src.final_detector import FinalReviewDetector
from src.stylometry_features import extract_stylometry_features


In [2]:
# Load the two source datasets
ai_df = pd.read_csv(repo_root / "ai_generated_tripadvisor_reviews_openai_diverse.csv")
human_df = pd.read_csv(repo_root / "tripadvisor_hotel_reviews.csv").dropna(subset=["Review"])

print(f"AI dataset: {len(ai_df):,} reviews")
print(f"Human dataset: {len(human_df):,} reviews")

# Quick sanity check on casing
ai_caps_per_review = ai_df["Review"].apply(lambda s: sum(c.isupper() for c in str(s))).mean()
human_caps_per_review = human_df["Review"].apply(lambda s: sum(c.isupper() for c in str(s))).mean()
print(f"\nAvg uppercase letters per review:")
print(f"  AI dataset (naturally cased):     {ai_caps_per_review:.1f}")
print(f"  Human dataset (Kaggle, lowercased): {human_caps_per_review:.1f}")

AI dataset: 5,434 reviews
Human dataset: 20,491 reviews



Avg uppercase letters per review:
  AI dataset (naturally cased):     9.1
  Human dataset (Kaggle, lowercased): 0.2


In [4]:
# Load the trained detector
detector = FinalReviewDetector()
print(f"Loaded detector: {detector.selected_model_name}")

Loaded detector: random_forest


**Experiment A** — remove casing from 500 AI reviews, compare predictions before and after.

In [5]:
# Sample 500 AI reviews deterministically
sample_ai = ai_df.sample(n=500, random_state=42).reset_index(drop=True)

def detect_text(text):
    feats = extract_stylometry_features(text)
    return detector.detect_review_dict(review_text=text, extracted_features=feats)

# Run on original (cased)
results_natural = [detect_text(t) for t in sample_ai["Review"]]
df_natural = pd.DataFrame(results_natural)

# Run on lowercased
results_lowered = [detect_text(t.lower()) for t in sample_ai["Review"]]
df_lowered = pd.DataFrame(results_lowered)

print("Experiment A results:")
print(f"  Natural-cased AI -> mean score = {df_natural['ai_likeness_score'].mean():5.1f}, "
      f"AI-predicted = {(df_natural['predicted_label']=='AI').sum()} / 500")
print(f"  Lowercased AI   -> mean score = {df_lowered['ai_likeness_score'].mean():5.1f}, "
      f"AI-predicted = {(df_lowered['predicted_label']=='AI').sum()} / 500")

flips_a = ((df_natural['predicted_label']=='AI') & (df_lowered['predicted_label']=='Human')).sum()
print(f"  AI -> Human flips: {flips_a} / 500 ({100*flips_a/500:.1f}%)")
print(f"  Score drop on lowercase: {df_natural['ai_likeness_score'].mean() - df_lowered['ai_likeness_score'].mean():.1f} points on average")

Experiment A results:
  Natural-cased AI -> mean score = 100.0, AI-predicted = 500 / 500
  Lowercased AI   -> mean score =  73.3, AI-predicted = 499 / 500
  AI -> Human flips: 1 / 500 (0.2%)
  Score drop on lowercase: 26.7 points on average


Casing alone drops mean score ~27 pts but flips only 0.2% of predictions — not the sole driver.

**Experiment B** — sentence-case 500 human reviews, check if any flip to AI.

In [7]:
def sentence_case(text):
    """Capitalize first letter of each sentence and standalone 'i'."""
    if not isinstance(text, str): return text
    text = text.strip()
    if not text: return text
    out = text[0].upper() + text[1:]
    out = re.sub(r"([.!?]\s+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), out)
    out = re.sub(r"\bi\b", "I", out)
    return out

sample_human = human_df.sample(n=500, random_state=42).reset_index(drop=True)
cased_human = sample_human["Review"].apply(sentence_case)

# Original
results_human_orig = [detect_text(t) for t in sample_human["Review"]]
df_human_orig = pd.DataFrame(results_human_orig)

# Sentence-cased
results_human_cased = [detect_text(t) for t in cased_human]
df_human_cased = pd.DataFrame(results_human_cased)

print("Experiment B results:")
print(f"  Original lowercased human -> mean score = {df_human_orig['ai_likeness_score'].mean():5.1f}, "
      f"AI-predicted = {(df_human_orig['predicted_label']=='AI').sum()} / 500")
print(f"  Sentence-cased human      -> mean score = {df_human_cased['ai_likeness_score'].mean():5.1f}, "
      f"AI-predicted = {(df_human_cased['predicted_label']=='AI').sum()} / 500")

flips_b = ((df_human_orig['predicted_label']=='Human') & (df_human_cased['predicted_label']=='AI')).sum()
print(f"  Human -> AI flips: {flips_b} / 500 ({100*flips_b/500:.1f}%)")

caps_orig = sample_human["Review"].apply(lambda s: sum(c.isupper() for c in str(s))).mean()
caps_cased = cased_human.apply(lambda s: sum(c.isupper() for c in str(s))).mean()
print(f"\n  Avg uppercase per review: original {caps_orig:.1f} -> sentence-cased {caps_cased:.1f}")
print(f"  Note: low transformation impact because the Kaggle corpus has few sentence-ending periods")

Experiment B results:
  Original lowercased human -> mean score =   0.0, AI-predicted = 0 / 500
  Sentence-cased human      -> mean score =   0.0, AI-predicted = 0 / 500
  Human -> AI flips: 0 / 500 (0.0%)

  Avg uppercase per review: original 0.2 -> sentence-cased 1.6
  Note: low transformation impact because the Kaggle corpus has few sentence-ending periods


No flips. The Kaggle corpus has little punctuation left, so capitalisation can't be re-added meaningfully.

**Experiment C** — stepwise transformation of a demo failure review to isolate which preprocessing step flips the prediction.

In [9]:
COMMON_STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "was", "are", "were",
    "be", "been", "being", "have", "has", "had", "do", "does", "did",
    "of", "to", "in", "for", "with", "on", "at", "by", "from", "as",
    "i", "we", "he", "she", "it", "they", "you", "me", "us", "him",
    "her", "them", "my", "our", "his", "their", "your",
    "this", "that", "these", "those", "there", "here",
    "will", "would", "could", "should", "may", "might", "can",
    "if", "then", "so", "than", "very", "just",
}

def lowercase_only(text): return text.lower()

def lowercase_no_punct(text): return re.sub(r"[^\w\s]", " ", text.lower())

def trip_style_keep_punct(text):
    text = text.lower()
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return " ".join(t for t in tokens if t not in COMMON_STOPWORDS)

def trip_style_no_punct(text):
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return " ".join(t for t in text.split() if t not in COMMON_STOPWORDS)

demo_review = (
    "We stayed here for four nights with two kids and a grandparent. The "
    "double-bed family room was tighter than the photos suggested but it "
    "worked. Walls are thin enough that we heard the hallway clearly during "
    "checkouts at 6am. Breakfast was hit and miss - the coffee machine kept "
    "breaking on day three and the pastries on day four were clearly from "
    "the previous morning. That said, the location is unbeatable - five "
    "minutes to the metro and ten to the old town. Staff were patient with "
    "our broken Italian. Would go back if the price was right but I would "
    "not pay rack rate."
)

real_human = human_df.iloc[42]["Review"]

variants = [
    ("1. demo Example 5: original (sentence-cased English)",          demo_review),
    ("2. demo Example 5 + lowercase only",                            lowercase_only(demo_review)),
    ("3. demo Example 5 + lowercase + no punct",                      lowercase_no_punct(demo_review)),
    ("4. demo Example 5 + lowercase + stopword-removed (keep punct)", trip_style_keep_punct(demo_review)),
    ("5. demo Example 5 + full TripAdvisor-style preprocessing",      trip_style_no_punct(demo_review)),
    ("6. real human review from corpus (untouched)",                  real_human),
]

rows = []
for label, txt in variants:
    feats = extract_stylometry_features(txt)
    result = detector.detect_review_dict(review_text=txt, extracted_features=feats)
    rows.append({
        "variant": label,
        "ai_likeness_score": result["ai_likeness_score"],
        "predicted_label": result["predicted_label"],
        "capital_letter_ratio": round(feats["capital_letter_ratio"], 4),
        "stopword_ratio": round(feats["stopword_ratio"], 4),
        "sentence_length_variance": round(feats["sentence_length_variance"], 2),
        "avg_word_length": round(feats["avg_word_length"], 2),
    })

attribution_df = pd.DataFrame(rows)
attribution_df

,variant,ai_likeness_score,predicted_label,capital_letter_ratio,stopword_ratio,sentence_length_variance,avg_word_length
0,1. demo Example 5: original (sentence-cased En...,100,AI,0.0158,0.394,25.55,4.38
1,2. demo Example 5 + lowercase only,68,AI,0.0000,0.394,25.55,4.38
2,3. demo Example 5 + lowercase + no punct,59,AI,0.0000,0.394,0.00,4.38
3,4. demo Example 5 + lowercase + stopword-remov...,0,Human,0.0000,0.000,9.96,5.46
4,5. demo Example 5 + full TripAdvisor-style pre...,0,Human,0.0000,0.000,0.00,5.46
5,6. real human review from corpus (untouched),0,Human,0.0000,0.083,0.00,5.21


Stopword removal causes the largest single flip (score 59 → 0). The OOD failure traces to all three Kaggle preprocessing operations jointly: lowercase + depunctuate + stopword-filter.

In [11]:
import pickle

outputs = {
    "experiment_a": {
        "sample_indices": sample_ai.index.tolist(),
        "natural_results": df_natural.to_dict(orient="records"),
        "lowered_results": df_lowered.to_dict(orient="records"),
        "natural_mean_score": float(df_natural["ai_likeness_score"].mean()),
        "lowered_mean_score": float(df_lowered["ai_likeness_score"].mean()),
        "ai_to_human_flips": int(flips_a),
    },
    "experiment_b": {
        "sample_indices": sample_human.index.tolist(),
        "original_results": df_human_orig.to_dict(orient="records"),
        "cased_results": df_human_cased.to_dict(orient="records"),
        "original_mean_score": float(df_human_orig["ai_likeness_score"].mean()),
        "cased_mean_score": float(df_human_cased["ai_likeness_score"].mean()),
        "human_to_ai_flips": int(flips_b),
    },
    "experiment_c": {
        "attribution_table": attribution_df.to_dict(orient="records"),
        "demo_review": demo_review,
        "real_human_review": real_human,
    },
}

out_path = repo_root / "models" / "casing_ood_experiment_outputs.pkl"
with open(out_path, "wb") as f:
    pickle.dump(outputs, f)

print(f"Saved results to {out_path}")
print(f"File size: {out_path.stat().st_size:,} bytes")

Saved results to ../models/casing_ood_experiment_outputs.pkl
File size: 2,134,592 bytes
